# Week 64, Day 1: From Bayesian linear regression to Gaussian processes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w064/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Weight-space Bayesian regression

Build 30 Gaussian bump features (width 0.25 radians) across orientation, compute the weight posterior, and the predictive mean and standard deviation of f on a grid. Only orientations in the first half of the range were tested.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
g0 = np.random.default_rng(641)
f_true = lambda t: 8 * np.exp(-((t - 1.0) / 0.4) ** 2) + 2
th = g0.uniform(0, 1.8, 12)                            # orientations tested, in radians
y = f_true(th) + g0.normal(0, 1.0, 12)
y = y - y.mean()                                        # centre, so a zero-mean prior is sensible
centres = np.linspace(0, np.pi, 30); M = len(centres)
feat = lambda t: np.exp(-0.5 * ((np.asarray(t)[:, None] - centres) / 0.25) ** 2)
alpha, s2 = 4.0, 1.0
Phi = feat(th)
Sigma_w = ...
m_w = Sigma_w @ Phi.T @ y / s2
grid = np.linspace(0, np.pi, 200)
Pg = feat(grid)
mean_pred = Pg @ m_w
sd_pred = np.sqrt(np.einsum("ij,jk,ik->i", Pg, Sigma_w, Pg))
plt.fill_between(grid, mean_pred - 2 * sd_pred, mean_pred + 2 * sd_pred, alpha=0.2)
plt.plot(grid, mean_pred); plt.plot(th, y, "o"); plt.xlabel("orientation (rad)"); plt.ylabel("centred response"); plt.show()
print(f"predictive sd: {sd_pred[np.argmin(np.abs(grid - 1.0))]:.2f} at 1 rad (tested), {sd_pred[-1]:.2f} at pi (untested)")

In [ ]:
import numpy as np
def test_ridge():
    "the posterior mean of the weights equals ridge regression with penalty sigma^2 / alpha"
    ridge = np.linalg.solve(Phi.T @ Phi + (s2 / alpha) * np.eye(M), Phi.T @ y)
    assert np.allclose(m_w, ridge)
def test_var():
    "the predictive standard deviation is smallest near the data and largest far from it"
    near = np.argmin(np.abs(grid - th[0])); far = np.argmin(np.abs(grid - np.pi))
    assert sd_pred[near] < sd_pred[far]

test_ridge()
test_var()
print("All checks passed")

### Exercise: The same predictions from the kernel

Form the kernel matrices K = alpha Phi Phi^T, K_star between grid and data, and the prior variances on the grid, and compute the predictive mean and standard deviation in function space.

In [ ]:
K = alpha * Phi @ Phi.T
Ks = alpha * Pg @ Phi.T                               # grid x data
kss = alpha * np.sum(Pg ** 2, axis=1)                 # prior variance at each grid point
A = np.linalg.solve(K + s2 * np.eye(len(th)), np.eye(len(th)))
mean_fs = ...
var_fs = kss - np.einsum("ij,jk,ik->i", Ks, A, Ks)
sd_fs = np.sqrt(np.maximum(var_fs, 0))
print("largest differences from the weight-space results:", np.abs(mean_fs - mean_pred).max(), np.abs(sd_fs - sd_pred).max())
print(f"matrix inverted: {K.shape} instead of {Sigma_w.shape}")

In [ ]:
import numpy as np
def test_same():
    "function-space mean and standard deviation equal the weight-space ones"
    assert np.allclose(mean_fs, mean_pred, atol=1e-8) and np.allclose(sd_fs, sd_pred, atol=1e-8)

test_same()
print("All checks passed")

### Exercise: The kernel of many bumps

For M bumps of width w = 0.25 spaced evenly over a wide range, with weight variance proportional to the spacing, compute the kernel between two inputs and compare it with the limit, a squared exponential with length-scale sqrt(2) w and the same value at zero distance.

In [ ]:
w = 0.25
dists = np.linspace(0, 1.5, 31)
errs = []
for M_ in (5, 20, 100, 1000):
    c_ = np.linspace(-6, 6, M_); spacing = c_[1] - c_[0]
    phi0 = np.exp(-0.5 * (c_ / w) ** 2)
    k_bumps = np.array([spacing * np.sum(phi0 * np.exp(-0.5 * ((dd - c_) / w) ** 2)) for dd in dists])
    k_limit = ...
    errs.append(np.abs(k_bumps - k_limit).max() / k_limit[0])
    print(f"M = {M_:4d}: largest relative difference from the squared exponential {errs[-1]:.4f}")

In [ ]:
import numpy as np
def test_converge():
    "the bump kernel approaches the squared exponential kernel as M grows"
    assert errs[-1] < 0.02 and errs[-1] < errs[0]

test_converge()
print("All checks passed")